# Package v1: merged AI job-risk table with O\*NET and BLS features

**Goal:** package the v1 table the same way [`package_v0.ipynb`](package_v0.ipynb) packages v0: an all-occupations
version with nothing dropped, a data dictionary for every column, and one Excel workbook.

**Reads:** `ai_risk_v1.csv` from [`Milestone_1.ipynb`](Milestone_1.ipynb) Task 9, the v0 package files it builds on,
and the two interim O\*NET and BLS tables. For the QA trace it also rereads the raw files.
**Writes** to `data/processed/milestone_1/v1/` (no v0 file is changed):

| File | Contents |
|---|---|
| `ai_risk_v1_all_occupations.csv` | All 808 occupations scored by *any* index, plus the O\*NET and BLS columns |
| `ai_risk_v1_data_dictionary.csv` | One row per column of both v1 tables: meaning, source, units |
| `ai_risk_v1.xlsx` | README, both tables, the coverage report and the dictionary, one sheet each |

**Not written:** a `percentile_ranks_v1` file and a `merged_ai_risk_v1` file. v1 has the same 693 occupations and
scores as v0, so its ranks are exactly `v0/percentile_ranks_v0.csv` (checked in §3), and `ai_risk_v1.csv` already is
the merged v1 table. Copies would only drift apart.

**Run order to reproduce:** `Milestone_1.ipynb` (Tasks 1-5) → `task7_percentile_ranks.ipynb` → `package_v0.ipynb`
→ `Milestone_1.ipynb` (Task 9) → this notebook.

## Setup

Runs on **Google Colab** (mounts the team Drive folder) or **locally** (uses the repo folder).

In [1]:
import os

import pandas as pd

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', None)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/BTT Fall AI Studio (swytch 1b)'
except ImportError:
    # Not on Colab: use the repo folder (this notebook sits in notebooks/, one level below it)
    BASE = os.path.dirname(os.getcwd())

RAW = f'{BASE}/data/raw'
INTERIM = f'{BASE}/data/interim'
V0 = f'{BASE}/data/processed/milestone_1/v0'
V1 = f'{BASE}/data/processed/milestone_1/v1'
print('Data folder:', f'{BASE}/data')

Data folder: C:\Users\Kamsi\Downloads\code\Swytch-1B-do-ai-job-risk-indices-agree/data


## 1. Inputs

In [2]:
INPUTS = {
    'ai_risk_v1': f'{V1}/ai_risk_v1.csv',
    'all_occupations_v0': f'{V0}/ai_risk_v0_all_occupations.csv',
    'dictionary_v0': f'{V0}/ai_risk_v0_data_dictionary.csv',
    'coverage_report_v0': f'{V0}/coverage_report_v0.csv',
    'percentile_ranks_v0': f'{V0}/percentile_ranks_v0.csv',
    'onet_features': f'{INTERIM}/onet_features_soc6.csv',
    'bls_projections': f'{INTERIM}/bls_projections_soc2018.csv',
}
for name, path in INPUTS.items():
    assert os.path.exists(path), f'{path} not found: run Milestone_1.ipynb (including Task 9) first'

# Read codes as text so '11' and '43-9021' are never turned into numbers
CODES_AS_TEXT = {'soc': str, 'major_group': str, 'bls_code': str}
v1 = pd.read_csv(INPUTS['ai_risk_v1'], dtype=CODES_AS_TEXT)
all_v0 = pd.read_csv(INPUTS['all_occupations_v0'], dtype=CODES_AS_TEXT)
dictionary_v0 = pd.read_csv(INPUTS['dictionary_v0'])
coverage = pd.read_csv(INPUTS['coverage_report_v0'], dtype=CODES_AS_TEXT)
ranks_v0 = pd.read_csv(INPUTS['percentile_ranks_v0'], dtype=CODES_AS_TEXT)
onet = pd.read_csv(INPUTS['onet_features'], dtype=CODES_AS_TEXT)
bls = pd.read_csv(INPUTS['bls_projections'], dtype=CODES_AS_TEXT)

tables = {'ai_risk_v1': v1, 'all_occupations_v0': all_v0, 'dictionary_v0': dictionary_v0,
          'coverage_report_v0': coverage, 'percentile_ranks_v0': ranks_v0, 'onet_features': onet,
          'bls_projections': bls}
for name, table in tables.items():
    print(f'{name:<20} {table.shape}')

ai_risk_v1           (693, 81)
all_occupations_v0   (808, 17)
dictionary_v0        (49, 6)
coverage_report_v0   (115, 5)
percentile_ranks_v0  (693, 10)
onet_features        (798, 37)
bls_projections      (867, 14)


## 2. All-occupations table (nothing dropped)

The v0 all-occupations table plus the same O\*NET and BLS columns that Task 9 added to `ai_risk_v1`. Occupations
missing from an index keep blank scores, exactly as in v0.

In [3]:
all_v1 = all_v0.merge(onet, on='soc', how='left', validate='one_to_one')
all_v1 = all_v1.merge(bls, on='soc', how='left', validate='one_to_one')

# Nothing may be added or lost, and the v0 columns must be unchanged
assert len(all_v1) == len(all_v0), 'row count changed'
pd.testing.assert_frame_equal(all_v1[all_v0.columns], all_v0)

# The 693 occupations in ai_risk_v1 must have the same O*NET and BLS values here as there
new_columns = []
for column in all_v1.columns:
    if column not in all_v0.columns:
        new_columns.append(column)
in_v1 = all_v1[all_v1['in_v0']].set_index('soc')[new_columns].sort_index()
pd.testing.assert_frame_equal(in_v1, v1.set_index('soc')[new_columns].sort_index())

print(f'Rows: {len(all_v1)} | new columns: {len(new_columns)}')
print('\nO*NET status (blank = no O*NET occupation for this code):')
print(all_v1['onet_status'].value_counts(dropna=False).to_string())
print('\nBLS match:')
print(all_v1['bls_match'].value_counts(dropna=False).to_string())

Rows: 808 | new columns: 49

O*NET status (blank = no O*NET occupation for this code):
onet_status
full features    774
job zone only     24
NaN               10

BLS match:
bls_match
direct               780
via combined code     27
not in BLS             1


## 3. Percentile ranks are unchanged from v0

v1 keeps v0's occupations and scores, so the `pct_` columns in `ai_risk_v1` must equal `percentile_ranks_v0.csv`.

In [4]:
rank_columns = []
for column in ranks_v0.columns:
    if column != 'soc':
        rank_columns.append(column)

ranks_in_v1 = v1[['soc'] + rank_columns].sort_values('soc').reset_index(drop=True)
ranks_saved = ranks_v0[['soc'] + rank_columns].sort_values('soc').reset_index(drop=True)
pd.testing.assert_frame_equal(ranks_in_v1, ranks_saved)
print(f'All {len(rank_columns)} rank columns in ai_risk_v1 match percentile_ranks_v0.csv for {len(ranks_saved)} occupations')

All 9 rank columns in ai_risk_v1 match percentile_ranks_v0.csv for 693 occupations


## 4. Data dictionary

The v0 columns keep their v0 descriptions (only the table names change). Every new column gets a row below; the
check at the end stops if any column is undocumented. Group names come from the O\*NET 30.3 Content Model Reference
(as in `eda_raw.ipynb` Part 5).

In [5]:
# Each O*NET aggregate: its domain and the Content Model groups it averages (same groups as Milestone_1 Task 9)
ONET_AGGREGATES = {
    'abl_verbal': ('Abilities', 'Verbal Abilities (1.A.1.a)'),
    'abl_reasoning': ('Abilities', 'Idea Generation and Reasoning Abilities (1.A.1.b)'),
    'abl_quantitative': ('Abilities', 'Quantitative Abilities (1.A.1.c)'),
    'abl_memory_perception': ('Abilities', 'Memory, Perceptual, Spatial Abilities and Attentiveness (1.A.1.d-g)'),
    'abl_fine_manipulation': ('Abilities', 'Fine Manipulative Abilities (1.A.2.a)'),
    'abl_control_reaction': ('Abilities', 'Control Movement and Reaction Time and Speed Abilities (1.A.2.b-c)'),
    'abl_physical': ('Abilities', 'Physical Strength, Endurance, and Flexibility, Balance and Coordination (1.A.3.a-c)'),
    'abl_visual': ('Abilities', 'Visual Abilities (1.A.4.a)'),
    'abl_auditory_speech': ('Abilities', 'Auditory and Speech Abilities (1.A.4.b)'),
    'skl_foundational': ('Skills', 'Foundational Areas, i.e. basic content skills (2.A.1)'),
    'skl_ways_of_working': ('Skills', 'Ways of Working, i.e. basic process skills (2.A.2)'),
    'skl_social': ('Skills', 'Social Skills (2.B.1)'),
    'skl_complex_problem_solving': ('Skills', 'Complex Problem Solving Skills (2.B.2)'),
    'skl_technical': ('Skills', 'Technical Skills (2.B.3)'),
    'skl_systems': ('Skills', 'Systems Skills (2.B.4)'),
    'skl_resource_mgmt': ('Skills', 'Resource Management Skills (2.B.5)'),
    'wa_get_information': ('Work Activities', 'Looking for and Receiving Job-Related Information (4.A.1.a)'),
    'wa_evaluate_information': ('Work Activities', 'Identifying and Evaluating Job-Relevant Information (4.A.1.b)'),
    'wa_data_processing': ('Work Activities', 'Information and Data Processing (4.A.2.a)'),
    'wa_reasoning_decisions': ('Work Activities', 'Reasoning and Decision Making (4.A.2.b)'),
    'wa_physical_manual': ('Work Activities', 'Performing Physical and Manual Work Activities (4.A.3.a)'),
    'wa_complex_technical': ('Work Activities', 'Performing Complex and Technical Activities (4.A.3.b)'),
    'wa_communicating': ('Work Activities', 'Communicating and Interacting (4.A.4.a)'),
    'wa_coordinating_managing': ('Work Activities', 'Coordinating, Developing, Managing, and Advising (4.A.4.b)'),
    'wa_administering': ('Work Activities', 'Administering (4.A.4.c)'),
    'wc_communication': ('Work Context', 'Communication (4.C.1.a)'),
    'wc_role_responsibility': ('Work Context', 'Role Relationships and Responsibility for Others (4.C.1.b-c)'),
    'wc_conflict': ('Work Context', 'Conflictual Contact (4.C.1.d)'),
    'wc_work_setting': ('Work Context', 'Work Setting (4.C.2.a)'),
    'wc_environment_hazards': ('Work Context', 'Environmental Conditions and Job Hazards (4.C.2.b-c)'),
    'wc_body_positioning': ('Work Context', 'Body Positioning (4.C.2.d)'),
    'wc_criticality': ('Work Context', 'Criticality of Position (4.C.3.a)'),
    'wc_routine_vs_challenging': ('Work Context', 'Routine versus Challenging Work (4.C.3.b)'),
    'wc_competition_pace': ('Work Context', 'Competition, and Pace and Scheduling (4.C.3.c-d)'),
}
SCALE = {'Abilities': 'Importance', 'Skills': 'Importance', 'Work Activities': 'Importance', 'Work Context': 'Context'}
SOURCE_FILE = {
    'Abilities': 'abilities.csv',
    'Skills': 'essential_skills.csv + transferable_skills.csv',
    'Work Activities': 'work_activities.csv',
    'Work Context': 'work_context.csv',
}

NEW_DESCRIPTIONS = {}  # column -> (description, source, units)
for aggregate, (domain, groups) in ONET_AGGREGATES.items():
    NEW_DESCRIPTIONS['onet_' + aggregate] = (
        f'O*NET {domain.lower()}: mean {SCALE[domain]} rating of the elements in {groups}',
        f'O*NET 30.3 {SOURCE_FILE[domain]}; Milestone_1 Task 9',
        f'{SCALE[domain]} scale 1-5',
    )

BLS_SOURCE = 'BLS Employment Projections 2025-35, Table 1.2 (occupation.xlsx); Milestone_1 Task 9'
NEW_DESCRIPTIONS.update({
    'onet_job_zone': ('O*NET Job Zone (preparation the job needs), averaged over the O*NET occupations in this SOC code',
                      'O*NET 30.3 job_zones.csv; Milestone_1 Task 9', 'Job Zone 1-5'),
    'onet_status': ("'full features', 'job zone only' (no O*NET ratings, so the onet_ features are blank), or blank if "
                    "O*NET has no occupation for this code", 'derived', 'text'),
    'bls_code': ('BLS National Employment Matrix code matched to this SOC code', BLS_SOURCE, 'code'),
    'bls_match': ("'direct', 'via combined code' (the BLS values are a total shared with other SOC codes: keep one row "
                  "per bls_code before summing or weighting), or 'not in BLS'", 'derived', 'text'),
    'bls_title': ('BLS occupation title for bls_code', BLS_SOURCE, 'text'),
    'bls_emp_2025': ('Employment in 2025', BLS_SOURCE, 'thousands of jobs'),
    'bls_emp_2035': ('Projected employment in 2035', BLS_SOURCE, 'thousands of jobs'),
    'bls_emp_change': ('Projected employment change, 2025-35', BLS_SOURCE, 'thousands of jobs'),
    'bls_emp_change_pct': ('Projected employment change, 2025-35', BLS_SOURCE, 'percent'),
    'bls_self_employed_pct': ('Share of 2025 employment that is self-employed', BLS_SOURCE, 'percent'),
    'bls_openings_per_year': ('Average yearly job openings, 2025-35', BLS_SOURCE, 'thousands of openings'),
    'bls_median_wage': ('Median annual wage, 2025', BLS_SOURCE, 'dollars'),
    'bls_education': ('Typical education needed for entry', BLS_SOURCE, 'text'),
    'bls_experience': ('Work experience in a related occupation needed for entry', BLS_SOURCE, 'text'),
    'bls_training': ('Typical on-the-job training needed to become competent', BLS_SOURCE, 'text'),
})

# v0 columns: reuse the v0 rows under the new table names
TABLE_NAMES = {'ai_risk_v0': 'ai_risk_v1', 'ai_risk_v0_all_occupations': 'ai_risk_v1_all_occupations'}
v0_rows = dictionary_v0.copy()
v0_rows['table'] = v0_rows['table'].map(TABLE_NAMES)

# New columns: one row each, per table
new_rows = []
for table_name, table, v0_columns in [('ai_risk_v1', v1, dictionary_v0.loc[dictionary_v0['table'] == 'ai_risk_v0', 'column']),
                                      ('ai_risk_v1_all_occupations', all_v1, all_v0.columns)]:
    for column in table.columns:
        if column in list(v0_columns):
            continue
        description, source, units = NEW_DESCRIPTIONS[column]
        dtype = str(table[column].dtype)
        if dtype == 'object':  # pandas 2 (Colab) calls text columns 'object', pandas 3 calls them 'str'
            dtype = 'str'
        new_rows.append({'table': table_name, 'column': column, 'description': description,
                         'source': source, 'units': units, 'dtype': dtype})

dictionary = pd.concat([v0_rows, pd.DataFrame(new_rows)], ignore_index=True)
# Keep each table's rows together, in the table's column order
dictionary = pd.concat([dictionary[dictionary['table'] == 'ai_risk_v1'],
                        dictionary[dictionary['table'] == 'ai_risk_v1_all_occupations']], ignore_index=True)

# Every column of both tables is documented, in order
assert list(dictionary.loc[dictionary['table'] == 'ai_risk_v1', 'column']) == list(v1.columns)
assert list(dictionary.loc[dictionary['table'] == 'ai_risk_v1_all_occupations', 'column']) == list(all_v1.columns)
print(dictionary.groupby('table').size().to_string())
dictionary[dictionary['column'].str.startswith(('onet_', 'bls_')) & (dictionary['table'] == 'ai_risk_v1')]

table
ai_risk_v1                    81
ai_risk_v1_all_occupations    66


,table,column,description,source,units,dtype
32,ai_risk_v1,onet_abl_verbal,O*NET abilities: mean Importance rating of the...,O*NET 30.3 abilities.csv; Milestone_1 Task 9,Importance scale 1-5,float64
33,ai_risk_v1,onet_abl_reasoning,O*NET abilities: mean Importance rating of the...,O*NET 30.3 abilities.csv; Milestone_1 Task 9,Importance scale 1-5,float64
34,ai_risk_v1,onet_abl_quantitative,O*NET abilities: mean Importance rating of the...,O*NET 30.3 abilities.csv; Milestone_1 Task 9,Importance scale 1-5,float64
35,ai_risk_v1,onet_abl_memory_perception,O*NET abilities: mean Importance rating of the...,O*NET 30.3 abilities.csv; Milestone_1 Task 9,Importance scale 1-5,float64
36,ai_risk_v1,onet_abl_fine_manipulation,O*NET abilities: mean Importance rating of the...,O*NET 30.3 abilities.csv; Milestone_1 Task 9,Importance scale 1-5,float64
37,ai_risk_v1,onet_abl_control_reaction,O*NET abilities: mean Importance rating of the...,O*NET 30.3 abilities.csv; Milestone_1 Task 9,Importance scale 1-5,float64
38,ai_risk_v1,onet_abl_physical,O*NET abilities: mean Importance rating of the...,O*NET 30.3 abilities.csv; Milestone_1 Task 9,Importance scale 1-5,float64
39,ai_risk_v1,onet_abl_visual,O*NET abilities: mean Importance rating of the...,O*NET 30.3 abilities.csv; Milestone_1 Task 9,Importance scale 1-5,float64
40,ai_risk_v1,onet_abl_auditory_speech,O*NET abilities: mean Importance rating of the...,O*NET 30.3 abilities.csv; Milestone_1 Task 9,Importance scale 1-5,float64
41,ai_risk_v1,onet_skl_foundational,O*NET skills: mean Importance rating of the el...,O*NET 30.3 essential_skills.csv + transferable...,Importance scale 1-5,float64


## 5. QA: trace three occupations back to the raw files

One occupation for each join situation. The O\*NET and BLS values are recomputed from the **raw** files (not the
interim ones) and compared with the packaged row:

1. **43-9021 Data Entry Keyers:** full O\*NET features, direct BLS match.
2. **13-1021 Buyers and Purchasing Agents, Farm Products:** BLS values come from the combined code `13-1020`.
3. **11-2032 Public Relations Managers:** O\*NET has a Job Zone but no ratings, so the features are blank.

In [6]:
QA_CODES = ['43-9021', '13-1021', '11-2032']
ONET_CODE = 'O*NET-SOC Code'

abilities_raw = pd.read_csv(f'{RAW}/abilities.csv', dtype={ONET_CODE: str})
job_zones_raw = pd.read_csv(f'{RAW}/job_zones.csv', dtype={ONET_CODE: str})
bls_raw = pd.read_excel(f'{RAW}/occupation.xlsx', sheet_name='Table 1.2', header=1,
                        keep_default_na=False, na_values=['—', ''])

packaged = v1.set_index('soc')
trace = []
for soc in QA_CODES:
    row = packaged.loc[soc]

    # Verbal abilities: Importance ratings of elements 1.A.1.a.*, averaged per element over the O*NET occupations, then over elements
    verbal = abilities_raw[(abilities_raw[ONET_CODE].str[:7] == soc)
                           & (abilities_raw['Scale ID'] == 'IM')
                           & (abilities_raw['Element ID'].str.startswith('1.A.1.a.'))]
    if len(verbal) > 0:
        verbal_expected = verbal.groupby('Element ID')['Data Value'].mean().mean()
    else:
        verbal_expected = float('nan')

    job_zone_expected = job_zones_raw.loc[job_zones_raw[ONET_CODE].str[:7] == soc, 'Job Zone'].mean()

    bls_row = bls_raw[bls_raw['2025 National Employment Matrix code'] == row['bls_code']]
    employment_expected = bls_row['Employment, 2025'].iloc[0]

    for column, expected in [('onet_abl_verbal', verbal_expected),
                             ('onet_job_zone', job_zone_expected),
                             ('bls_emp_2025', employment_expected)]:
        trace.append({'soc': soc, 'title': row['title'], 'bls_code': row['bls_code'], 'value': column,
                      'recomputed from raw': expected, 'packaged': row[column]})

trace = pd.DataFrame(trace)
both_blank = trace['recomputed from raw'].isna() & trace['packaged'].isna()
close = (trace['recomputed from raw'] - trace['packaged']).abs() < 1e-9
trace['match'] = both_blank | close
trace

,soc,title,bls_code,value,recomputed from raw,packaged,match
0,43-9021,Data Entry Keyers,43-9021,onet_abl_verbal,3.3750,3.3750,True
1,43-9021,Data Entry Keyers,43-9021,onet_job_zone,2.0000,2.0000,True
2,43-9021,Data Entry Keyers,43-9021,bls_emp_2025,131.8000,131.8000,True
3,13-1021,"Buyers and Purchasing Agents, Farm Products",13-1020,onet_abl_verbal,3.6225,3.6225,True
4,13-1021,"Buyers and Purchasing Agents, Farm Products",13-1020,onet_job_zone,4.0000,4.0000,True
5,13-1021,"Buyers and Purchasing Agents, Farm Products",13-1020,bls_emp_2025,516.1000,516.1000,True
6,11-2032,Public Relations Managers,11-2032,onet_abl_verbal,NaN,NaN,True
7,11-2032,Public Relations Managers,11-2032,onet_job_zone,4.0000,4.0000,True
8,11-2032,Public Relations Managers,11-2032,bls_emp_2025,80.8000,80.8000,True


In [7]:
# Stop before exporting if any trace failed
assert trace['match'].all(), trace[~trace['match']]
print(f'QA trace passed: {len(trace)} values across {len(QA_CODES)} occupations match the raw files')

QA trace passed: 9 values across 3 occupations match the raw files


## 6. Write the package

In [8]:
readme = pd.DataFrame({'README': [
    'AI job-risk indices with O*NET and BLS features, merged on SOC 2018 (Swytch 1B, Milestone 1, v1)',
    '',
    'Sheets:',
    f'  ai_risk_v1: {len(v1)} occupations scored by all three indices, with percentile ranks, provenance flags, 34 O*NET aggregates, Job Zone and BLS 2025-35 projections',
    f'  all_occupations: all {len(all_v1)} occupations scored by any index, with the same O*NET and BLS columns; missing values left blank',
    f'  coverage_report: the {len(coverage)} occupations not in ai_risk_v1 and which index is missing (unchanged from v0)',
    '  data_dictionary: meaning, source and units of every column',
    '',
    'Scales differ (share / z-score / probability): compare indices using the pct_ columns only.',
    "Employment: rows with bls_match = 'via combined code' repeat a BLS total; keep one row per bls_code before summing or weighting.",
    'O*NET features are blank where onet_status is not "full features".',
    'Built by notebooks/package_v1.ipynb from Milestone_1.ipynb Task 9.',
]})

os.makedirs(V1, exist_ok=True)
all_v1.to_csv(f'{V1}/ai_risk_v1_all_occupations.csv', index=False)
dictionary.to_csv(f'{V1}/ai_risk_v1_data_dictionary.csv', index=False)
with pd.ExcelWriter(f'{V1}/ai_risk_v1.xlsx') as workbook:
    readme.to_excel(workbook, sheet_name='README', index=False)
    v1.to_excel(workbook, sheet_name='ai_risk_v1', index=False)
    all_v1.to_excel(workbook, sheet_name='all_occupations', index=False)
    coverage.to_excel(workbook, sheet_name='coverage_report', index=False)
    dictionary.to_excel(workbook, sheet_name='data_dictionary', index=False)

for file_name in sorted(os.listdir(V1)):
    size_kb = os.path.getsize(f'{V1}/{file_name}') / 1024
    print(f'{file_name:<40} {size_kb:8.1f} KB')

ai_risk_v1.csv                              621.3 KB
ai_risk_v1.xlsx                             644.3 KB
ai_risk_v1_all_occupations.csv              579.4 KB
ai_risk_v1_data_dictionary.csv               26.8 KB


In [9]:
# Round-trip check: the saved files read back with the same shape and columns
all_back = pd.read_csv(f'{V1}/ai_risk_v1_all_occupations.csv')
assert all_back.shape == all_v1.shape and list(all_back.columns) == list(all_v1.columns)
workbook = pd.ExcelFile(f'{V1}/ai_risk_v1.xlsx')
print('sheets:', workbook.sheet_names)
assert workbook.parse('ai_risk_v1').shape == v1.shape
assert workbook.parse('all_occupations').shape == all_v1.shape
print('round-trip OK')

sheets: ['README', 'ai_risk_v1', 'all_occupations', 'coverage_report', 'data_dictionary']


round-trip OK


## Result and retained limitations

- **`ai_risk_v1_all_occupations`:** all 808 occupations, none dropped, with the 49 O\*NET and BLS columns. Use it
  when a question does not need all three indices (e.g. case studies of occupations one index leaves out).
- **`ai_risk_v1_data_dictionary`:** every column of both v1 tables. The v0 columns keep their v0 descriptions.
- **`ai_risk_v1.xlsx`:** both tables, the coverage report and the dictionary, with a README sheet.
- **Percentile ranks** are unchanged from v0 (checked in §3); read them from `ai_risk_v1` or `v0/percentile_ranks_v0.csv`.
- Three occupations were traced back to the raw O\*NET and BLS files before export.
- **Retained limitations:** those listed at the end of `Milestone_1.ipynb` Task 9 (O\*NET vintage, unflagged
  partial O\*NET coverage, mixed-direction aggregates, BLS combined codes), plus all v0 limitations in
  `package_v0.ipynb`.